# 0 - Setup

In [1]:
%%capture
!pip install -q langchain langchain-community langchain-google-genai langchain-chroma chromadb gradio pypdf pdfplumber langchain-text-splitters sentence-transformers langchain-huggingface huggingface_hub
!pip install -U sentence-transformers langchain-huggingface huggingface_hub --quiet

print('installs done')


In [2]:
import warnings
warnings.filterwarnings('ignore')
import os, pathlib
import pdfplumber
print('imports ok')


imports ok


In [3]:
from google.colab import userdata
import os
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('Key stored')


Key stored


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings

llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite', temperature=0)
# embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

# Using a local, lite, and fast embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# quick check
response = llm.invoke("Say hello in 3 words")

if isinstance(response.content, list):
    text = "".join(
        block.get("text", "") for block in response.content
        if isinstance(block, dict) and block.get("type") == "text"
    )
else:
    text = response.content

print(text)

/tmp/ipykernel_19524/4063864739.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Hello there, friend!


---
# Level 1 - Bring Your Own PDF


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document

vectorstore = None
chunks_global = []
docs_global = []
current_pdf_name = None

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

def load_with_pypdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    return loader.load()

def ingest_pdf(pdf_path, chunk_size=1000, chunk_overlap=200, loader_fn=None):
    global vectorstore, chunks_global, docs_global, current_pdf_name
    if loader_fn is None:
        loader_fn = load_with_pypdf
    docs = loader_fn(pdf_path)
    print(f'Loaded {len(docs)} pages')
    chunks = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap).split_documents(docs)
    print(f'Split into {len(chunks)} chunks')
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
    chunks_global, docs_global = chunks, docs
    current_pdf_name = pathlib.Path(pdf_path).name
    print(f'Stored {len(chunks)} vectors - ready to chat')
    return vectorstore, chunks, docs


---
# Level 2 - Handle Messy PDFs


In [6]:
import pdfplumber
import pathlib
import logging

logger = logging.getLogger(__name__)

def _table_to_markdown(table, t_idx, page_num):
    if not table or len(table) == 0:
        return ''

    # Filter out fully-empty rows
    table = [row for row in table if row and any(c not in (None, '') for c in row)]
    if not table:
        return ''

    header = table[0]
    rows = table[1:]
    n_cols = len(header)

    def clean_row(row, n_cols):
        # Pad or truncate rows to match header length
        row = [str(c).strip().replace('\n', ' ').replace('|', '/') if c is not None else '' for c in row]
        if len(row) < n_cols:
            row = row + [''] * (n_cols - len(row))
        elif len(row) > n_cols:
            row = row[:n_cols]
        return row

    md_header = ' | '.join(clean_row(header, n_cols))
    md_sep = ' | '.join('---' for _ in range(n_cols))
    md_rows = [' | '.join(clean_row(r, n_cols)) for r in rows if r]

    if not md_rows:
        return ''

    return f"\n\n[Table {t_idx+1} on page {page_num}]\n{md_header}\n{md_sep}\n" + "\n".join(md_rows)


def load_with_pdfplumber(pdf_path, min_chars_for_ocr_warning=20):
    docs = []
    skipped_pages = []
    likely_scanned_pages = []

    try:
        pdf = pdfplumber.open(pdf_path)
    except Exception as e:
        logger.error(f"Failed to open PDF '{pdf_path}': {e}")
        print(f"[ERROR] Could not open {pdf_path}: {e}")
        return docs  # return empty list rather than crashing the whole pipeline

    try:
        with pdf:
            total_pages = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                page_num = i + 1
                text = ''
                tables = []

                # Text extraction — isolate failure per page
                try:
                    text = page.extract_text() or ''
                except Exception as e:
                    logger.warning(f"Text extraction failed on page {page_num} of {pdf_path}: {e}")
                    skipped_pages.append((page_num, 'text_extract_error', str(e)))

                # Table extraction — isolate failure per page
                try:
                    tables = page.extract_tables() or []
                except Exception as e:
                    logger.warning(f"Table extraction failed on page {page_num} of {pdf_path}: {e}")
                    skipped_pages.append((page_num, 'table_extract_error', str(e)))

                # Build table markdown, skipping any single bad table rather than
                # letting one malformed table kill the whole page
                table_text = ''
                for t_idx, table in enumerate(tables):
                    try:
                        table_text += _table_to_markdown(table, t_idx, page_num)
                    except Exception as e:
                        logger.warning(f"Failed to render table {t_idx+1} on page {page_num}: {e}")
                        continue

                page_content = (text + table_text).strip()

                if not page_content:
                    # Could be a blank page or an image-only (scanned) page
                    likely_scanned_pages.append(page_num)
                    continue

                # Flag pages with suspiciously little text — often signals
                # partially-scanned pages, watermarked pages, or extraction issues
                if len(text.strip()) < min_chars_for_ocr_warning and not table_text:
                    likely_scanned_pages.append(page_num)

                docs.append(Document(
                    page_content=page_content,
                    metadata={
                        'source': pathlib.Path(pdf_path).name,
                        'page': i,
                        'page_number': page_num,
                        'total_pages': total_pages,
                        'has_tables': bool(tables),
                        'n_tables': len(tables),
                    }
                ))

    except Exception as e:
        # Catch-all so one unexpected error doesn't lose docs already extracted
        logger.error(f"Unexpected error while processing '{pdf_path}': {e}")
        print(f"[ERROR] Unexpected failure processing {pdf_path}: {e}")

    print(f"pdfplumber loaded {len(docs)}/{total_pages if 'total_pages' in dir() else '?'} pages from {pathlib.Path(pdf_path).name}")
    if skipped_pages:
        print(f"   {len(skipped_pages)} page(s) had extraction errors: {[p[0] for p in skipped_pages]}")
    if likely_scanned_pages:
        print(f"   {len(likely_scanned_pages)} page(s) had little/no text — possibly scanned/image-only: {likely_scanned_pages}")

    return docs

---
# Level 3 - Streaming


In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
{context}

Question: {question}

Answer:"""
)

REWRITE_PROMPT = ChatPromptTemplate.from_template(
    'Rewrite the follow-up as a standalone question using history.\nHistory:\n{history}\nFollow-up: {question}\nStandalone:'
)

def _extract_content(content):
    """Gemini via langchain may return content as str or list of blocks; always return str."""
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts=[]
        for p in content:
            if isinstance(p, str):
                parts.append(p)
            elif isinstance(p, dict):
                parts.append(p.get('text','') or p.get('content','') or '')
            else:
                parts.append(getattr(p, 'text', str(p)))
        return ''.join(parts)
    return str(content)

def _format_history(history):
    if not history:
        return 'no history'
    return '\n'.join([f'User: {u}\nAssistant: {a}' for u, a in history[-3:]])

def ask(question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = '\n\n'.join(d.page_content for d in docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    raw = llm.invoke(prompt).content
    print(f'[LLM raw] {raw!r}')  # visible in Colab console and test_rag
    return _extract_content(raw)

def ask_stream(question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = '\n\n'.join(d.page_content for d in docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    acc = ''
    for chunk in llm.stream(prompt):
        delta = _extract_content(chunk.content)
        print(f'[LLM chunk] {delta!r} | raw={chunk.content!r}')
        acc += delta
        yield acc


---
# Level 4 - Citations
Show `source` and `page` from `chunk.metadata` after each answer.


In [8]:
def ask_with_citations(question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = '\n\n'.join(d.page_content for d in docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    raw = llm.invoke(prompt).content
    print(f'[LLM raw] {raw!r}')
    answer = _extract_content(raw)
    cites = []
    for d in docs:
        cites.append(f"{d.metadata.get('source', current_pdf_name)} p.{d.metadata.get('page', 0)+1}")
    cites = list(dict.fromkeys(cites))
    cite_str = '\n'.join('- ' + c for c in cites)
    return answer + '\n\n**Sources:**\n' + cite_str


---
# Level 5 - Conversational
Make the bot remember history so "he/she/it" works.


In [9]:
CONV_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
{context}

Question: {question}

Answer:"""
)

def rewrite_question(question, history):
    if not history:
        return question
    prompt = REWRITE_PROMPT.format(history=_format_history(history), question=question)
    raw = llm.invoke(prompt).content
    txt = _extract_content(raw).strip()
    print(f'[rewrite] {question!r} -> {txt!r} | raw={raw!r}')
    return txt

def ask_conversational_stream(message, history, k=3):
    standalone = rewrite_question(message, history)
    docs = vectorstore.similarity_search(standalone, k=k)
    context = '\n\n'.join(d.page_content for d in docs)
    prompt = CONV_PROMPT.format(context=context, history=_format_history(history), question=standalone)
    print(f'[RAG] standalone={standalone!r} k={k} docs={len(docs)}')
    # show retrieved context preview in console so you see what LLM sees
    print(f'[RAG context preview] {context[:600]!r}')
    cites = list(dict.fromkeys([f"{d.metadata.get('source', current_pdf_name)} p.{d.metadata.get('page', 0)+1}" for d in docs]))
    footer = '\n\n**Sources:**\n' + '\n'.join('- ' + c for c in cites)
    acc = ''
    for chunk in llm.stream(prompt):
        delta = _extract_content(chunk.content)
        print(f'[LLM chunk] {delta!r} | raw={chunk.content!r}')
        acc += delta
        yield acc + footer


---
# Gradio Chatbot (Levels 1-5)
Upload PDF, pick loader, chat with streaming + citations + history.


In [10]:
import gradio as gr
import traceback

def on_upload(file, loader_choice, chunk_size, chunk_overlap):
    try:
        if file is None:
            return 'No file selected. Drop a PDF.'
        pdf_path = file.name
        loader_fn = load_with_pdfplumber if 'pdfplumber' in loader_choice else load_with_pypdf
        ingest_pdf(pdf_path, chunk_size=int(chunk_size), chunk_overlap=int(chunk_overlap), loader_fn=loader_fn)
        return f'Loaded {current_pdf_name} - {len(chunks_global)} chunks. You can now chat below.'
    except Exception as e:
        traceback.print_exc()
        return f'Load failed: {e}\n' + traceback.format_exc()[-700:]

def _to_tuples(history):
    """Convert Gradio history (messages or tuples) to [(user, assistant), ...] for RAG."""
    if not history:
        return []
    if isinstance(history[0], dict):
        tuples = []
        for i in range(0, len(history)-1, 2):
            if history[i].get('role')=='user' and history[i+1].get('role')=='assistant':
                tuples.append((history[i]['content'], history[i+1]['content']))
        return tuples
    return [(u, a) for u, a in history]

def chat_stream(message, history, k, show_raw=False):
    """Streaming chat that works with Gradio 5 messages format; yields history and optional raw."""
    if history is None:
        history = []
    is_messages = len(history)==0 or isinstance(history[0], dict)
    if not is_messages:
        history = [m for pair in history for m in [{'role':'user','content':pair[0]}, {'role':'assistant','content':pair[1]}]]
    raw_acc = ''
    try:
        if vectorstore is None:
            history = history + [{'role':'user','content':message}, {'role':'assistant','content':'No PDF loaded yet. Upload a PDF first (top panel).'}]
            yield history, ''
            return
        if not message or not message.strip():
            history = history + [{'role':'user','content':message}, {'role':'assistant','content':'Please type a question.'}]
            yield history, ''
            return
        history = history + [{'role':'user','content':message}, {'role':'assistant','content':''}]
        prior_tuples = _to_tuples(history[:-2])
        for chunk in ask_conversational_stream(message, prior_tuples, k=int(k)):
            history[-1]['content'] = chunk
            # raw = chunk without Sources footer for debug pane
            raw_acc = chunk.split('\n\n**Sources:**')[0]
            yield history, raw_acc
    except Exception as e:
        traceback.print_exc()
        err = f'Error: {e}\n' + traceback.format_exc()[-800:]
        if history and history[-1].get('role')=='assistant':
            history[-1]['content'] = err
        else:
            history = history + [{'role':'assistant','content':err}]
        yield history, err

with gr.Blocks(title='RAG Chatbot') as demo:
    gr.Markdown('# RAG Chatbot - Upload a PDF and chat')
    gr.Markdown('Tip: after upload run the Verification cell. Raw LLM chunks are printed to the Colab console and shown in the Raw box below.')
    with gr.Row():
        with gr.Column():
            file_in = gr.File(label='Drop PDF', file_types=['.pdf'])
            loader_choice = gr.Radio(['PyPDFLoader (fast)', 'pdfplumber (tables)'], value='pdfplumber (tables)', label='Loader')
            chunk_size = gr.Slider(500, 2000, value=1000, step=100, label='chunk_size')
            chunk_overlap = gr.Slider(0, 400, value=200, step=20, label='chunk_overlap')
            status = gr.Markdown('No PDF loaded yet')
        with gr.Column():
            try:
                chatbot = gr.Chatbot(label='Chat (with citations)', height=380, type='messages')
            except TypeError:
                chatbot = gr.Chatbot(label='Chat (with citations)', height=380)
            raw_box = gr.Textbox(label='Raw LLM (without footer) - live', lines=4, max_lines=8)
            msg = gr.Textbox(label='Your question', placeholder="Ask about the PDF, try follow-up with 'he' ")
            k_slider = gr.Slider(1, 5, value=3, step=1, label='k')
    file_in.change(on_upload, inputs=[file_in, loader_choice, chunk_size, chunk_overlap], outputs=status)
    msg.submit(chat_stream, inputs=[msg, chatbot, k_slider], outputs=[chatbot, raw_box])
    msg.submit(lambda: '', outputs=msg)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cbfa7b2783e989f341.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Quick check after upload
Run this to see if ingest worked. Try changing `k` and `chunk_size` and see how retrieval changes.


In [11]:
print(f'Docs: {len(docs_global)}, Chunks: {len(chunks_global)}')
if chunks_global:
    print(chunks_global[0].metadata)
    print(chunks_global[0].page_content[:300])


Docs: 0, Chunks: 0


In [12]:
# Verification block: PDF -> text -> chunks -> embeddings -> Chroma
# Run this AFTER uploading a PDF. Shows counts, embedding dim, and a retrieval sample.

def verify_ingest(test_q='What is this document about?', k=3):
    import traceback
    if vectorstore is None or not chunks_global:
        print('No vectors yet. Upload a PDF first (status should say Loaded ...).')
        print(f'  vectorstore is None: {vectorstore is None}')
        print(f'  len(chunks_global): {len(chunks_global)}')
        return
    print(f'Docs (pages): {len(docs_global)}')
    print(f'Chunks: {len(chunks_global)}')
    print(f'Current file: {current_pdf_name}')
    print(f'Chunk 0 meta: {chunks_global[0].metadata}')
    print(f'Chunk 0 preview: {chunks_global[0].page_content[:300]!r}')
    try:
        n = vectorstore._collection.count()
        print(f'Chroma vectors: {n}  (expected {len(chunks_global)})')
        if n != len(chunks_global):
            print('  Mismatch - re-run ingest_pdf')
    except Exception as e:
        print(f'[Chroma count] failed: {e}')
        traceback.print_exc()
    try:
        v = embeddings.embed_query('hello world')
        print(f'Embeddings ok: dim={len(v)}')
    except Exception as e:
        print(f'Embeddings FAILED: {e}')
        traceback.print_exc()
        print(' Hint: check GOOGLE_API_KEY / quota at https://aistudio.google.com/apikey')
        return
    try:
        hits = vectorstore.similarity_search(test_q, k=k)
        print(f'Retrieval ok: {len(hits)} hits for q={test_q!r}')
        for i, h in enumerate(hits):
            print(f'  hit {i+1}: {h.metadata.get("source", current_pdf_name)} p.{h.metadata.get("page",0)+1} | {h.page_content[:120]!r}')
    except Exception as e:
        print(f'Retrieval FAILED: {e}')
        traceback.print_exc()

# run now (prints No vectors yet until you upload):
verify_ingest()


No vectors yet. Upload a PDF first (status should say Loaded ...).
  vectorstore is None: True
  len(chunks_global): 0
